# 06. Findings and Recommendations

Synthesis of notebooks 03-05. This is a write-up notebook — minimal code,
mostly markdown. All numbers below are pulled from
`results/threshold_sweep.csv`, `results/model_comparison_summary.csv`,
and `results/boost_ablation.csv`.


In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

t_sweep = pd.read_csv(ROOT / "results" / "threshold_sweep.csv")
model_summ = pd.read_csv(ROOT / "results" / "model_comparison_summary.csv")
ablation = pd.read_csv(ROOT / "results" / "boost_ablation.csv", index_col=0)

best_t = t_sweep.loc[t_sweep["f1"].idxmax()]
print(f"Best threshold (e5-base, this corpus): T={best_t['threshold']:.2f}  F1={best_t['f1']:.3f}")
print()
print("Per-model best F1 + threshold:")
print(model_summ[["model", "threshold", "f1", "ingest_time_s"]].round(3).to_string(index=False))
print()
print("Boost ablation @ best T (e5-base):")
print(ablation.round(3))


Best threshold (e5-base, this corpus): T=0.81  F1=0.201

Per-model best F1 + threshold:
                         model  threshold    f1  ingest_time_s
         cl-nagoya/ruri-v3-30m       0.86 0.246          2.444
        cl-nagoya/ruri-v3-310m       0.84 0.327         23.863
 intfloat/multilingual-e5-base       0.82 0.182          7.872
intfloat/multilingual-e5-large       0.80 0.268         23.758

Boost ablation @ best T (e5-base):
                             precision  recall     f1  avg_returned  \
condition                                                             
all on (default)                 0.159   0.649  0.201        13.179   
no severity boost                0.159   0.649  0.201        13.179   
no applicability penalty         0.067   0.804  0.111        40.821   
no query expand                  0.082   0.929  0.133        33.571   
vanilla (none of the above)      0.030   1.000  0.058        69.036   

                             adv_empty_rate  
condition        

## Headline

On a corpus of **15 Japanese technical-doc markdown files (127 H2 chunks)**
and **32 hand-authored evaluation queries** (28 in-domain, 4 adversarial),
two clear findings emerge:

1. **agent-kms's documented default of `score_threshold=0.93` does not
   generalize.** On this corpus, at T=0.93 every in-domain query
   returns zero chunks (100% empty-rate). The actual best F1 with the
   default model `multilingual-e5-base` lands at **T = 0.81, F1 = 0.20**.

2. **The default embedding model `multilingual-e5-base` is the weakest
   of the four tested.** Swapping it for `cl-nagoya/ruri-v3-310m`
   (Japanese-specialized, same 768d) lifts best F1 from 0.18 to **0.33**
   — a ~80% relative improvement, with no change to retrieval logic
   and similar ingest cost.

Together, these findings imply that an agent-kms user who follows the
README defaults on a Japanese public-documentation corpus will get
**100% empty results** until they calibrate the threshold, and even
then will be ~40-80% below the F1 ceiling reachable by switching the
embedding model.

The boost machinery (`severity`, `applicability`, `query_expand`) has
secondary effects: severity is a no-op for uncurated corpora,
applicability penalty acts as a hidden -0.015 score shift, and query
expansion is the most useful of the three for paraphrased queries.


## Recommendations

### For agent-kms upstream

1. **Reconsider the default embedding model.** `intfloat/multilingual-e5-base`
   is the weakest of four models tested on this Japanese-doc corpus.
   `cl-nagoya/ruri-v3-310m` produces ~80% higher F1 at the same vector
   dimensionality (768d) and comparable ingest cost.

2. **Document the corpus-dependence of `score_threshold=0.93` prominently.**
   The README presents 0.93 as the empirically-chosen sweet spot; in
   practice it returns zero chunks on at least one realistic corpus.
   A short paragraph and a "if your retrieve calls return nothing, try
   T = 0.80 - 0.85 first" hint would prevent silent failure.

3. **Add a `calibrate-threshold` CLI subcommand.** Given a small labelled
   set (e.g. 20 query → expected-source pairs), it would sweep
   thresholds and propose the F1-optimal one for the user's specific
   corpus + model.

4. **Make the applicability penalty opt-in for uncurated corpora.**
   Today, ingesting a corpus without explicit `applicability` tags
   silently applies a -0.015 score shift to every chunk — this is the
   kind of behavior that makes the threshold default feel unreliable.

5. **Surface the score distribution in `agent-kms ingest`.** Print e.g.
   "p90 cosine for sample queries: 0.85; consider T ≤ 0.82". This
   would prevent users from silently getting empty results.

### For agent-kms users

- **Sweep before trusting any default.** The sweep code in this study
  generalises to any corpus in <100 lines (`src/eval_utils.py` + a YAML).
- **If you switch the embedding model, your threshold must move with it.**
  Per-model best thresholds in this study span 0.80 – 0.86 — a 0.06
  range that is bigger than the F1 differences between models.
- **Severity / applicability tagging is leverage you have to opt in to.**
  For a flat (untagged) corpus, the severity boost is a no-op and the
  applicability penalty acts as a uniform -0.015 score shift — equivalent
  to lowering your threshold by that much. Tag chunks explicitly or
  disable the boosts.


## Limitations

- **N=32 queries is small.** Confidence intervals on F1 are wide;
  consider a single decimal of F1 difference noise.
- **Single annotator** (myself). Gold relevance judgements reflect
  one person's view of what should be retrieved for each query.
- **One corpus domain.** React + JS Primer are intentionally similar
  registers (technical Japanese, web frameworks). Findings about
  threshold magnitude may not transfer to e.g. enterprise wikis,
  scientific papers, or legal Japanese.
- **Single point in time.** Embedding models, agent-kms's defaults, and
  the corpus content all evolve. Re-run this study any time agent-kms
  changes its default threshold or model.
- **No end-to-end LLM evaluation.** We measure retrieval quality, not
  whether the retrieved context lets an LLM produce a correct answer.


## Future work

- Expand to 100+ queries with at least 2 annotators per query;
  publish inter-annotator agreement.
- Add an **end-to-end LLM-as-judge** stage that runs Claude / Gemini on
  retrieved chunks and scores the final answer.
- Evaluate **hybrid retrieval** (BM25 + dense) — agent-kms's `sparse.py`
  module already implements the sparse half; this study covers only the
  dense path.
- Extend the corpus to include enterprise-style heterogeneous knowledge
  (FAQ + ADRs + slack snippets) to test agent-kms's `severity` /
  `applicability` machinery in the regime it was actually designed for.
